# PI-RAFT AMV Retrieval Pipeline Orchestration Shell

This notebook acts as the orchestration and experimentation shell for the **Physics-Informed Recurrent All-Pairs Field Transform (PI-RAFT)** pipeline. All core logic (loaders, layers, loss functions, exports, plotting) has been modularized into standard packages.

In [ ]:
# Install required libraries
!pip install boto3 netcdf4 xarray h5py

In [ ]:
!git clone https://github.com/ice-user/PI-RAFT-INSAT.git
%cd PI-RAFT-INSAT
!pip install -r requirements.txt
!pip install -e .

In [ ]:
import os
import sys
import torch
from torch.utils.data import ConcatDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

# Add parent directory to enable importing packages
sys.path.append(os.path.abspath(".."))

from datasets import SatelliteDataset
from models import PhysicsInformedRAFT
from losses import PhysicsInformedLoss
from evaluation import (
    plot_untrained_baseline,
    plot_inference_scaling_comparison,
    plot_netcdf_verification
)
from export import export_amv_netcdf

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Running pipeline on device context: {device}")

print("Initializing multi-hour GOES-16 proxy data streams...")
hours_to_fetch = [14, 15, 16, 17]

hourly_datasets = [
    SatelliteDataset(
        satellite='GOES16', product='ABI-L2-CMIPC', band='C09',
        year=2024, day_of_year=120, hour=h,
        sequence_length=4
    )
    for h in hours_to_fetch
]

sequence_dataset = ConcatDataset(hourly_datasets)
print(f"Total available tracking frames in sequence pool: {len(sequence_dataset)}")
sequence_loader = DataLoader(sequence_dataset, batch_size=1, shuffle=False)

print("Initializing unseen validation GOES-16 proxy data stream...")
val_dataset = SatelliteDataset(
    satellite='GOES16', product='ABI-L2-CMIPC', band='C09',
    year=2024, day_of_year=120, hour=18,
    sequence_length=4
)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)
print(f"Total validation tracking frames compiled: {len(val_dataset)}")

model = PhysicsInformedRAFT().to(device)
criterion = PhysicsInformedLoss(alpha=0.5, beta=0.1, gamma=0.01, epsilon=0.001)


In [ ]:
print("Generating Untrained Baseline Vector Field...")
images, dem = next(iter(sequence_loader))
images, dem = images.to(device), dem.to(device)

untrained_model = PhysicsInformedRAFT().to(device)
untrained_model.eval()

with torch.no_grad():
    flow_pred, _ = untrained_model(images, dem, iters=4)

plot_untrained_baseline(images[:, 0], flow_pred, stride=16)

In [ ]:
import torch.optim as optim

def save_checkpoint(epoch_num, model, optimizer, metrics):
    checkpoint_dir = "./model_checkpoints"
    os.makedirs(checkpoint_dir, exist_ok=True)
    optimizer_state_on_cpu = {}
    for k, v in optimizer.state_dict().items():
        if isinstance(v, dict):
            optimizer_state_on_cpu[k] = {}
            for sub_k, sub_v in v.items():
                if isinstance(sub_v, torch.Tensor):
                    optimizer_state_on_cpu[k][sub_k] = sub_v.cpu()
                else:
                    optimizer_state_on_cpu[k][sub_k] = sub_v
        elif isinstance(v, torch.Tensor):
            optimizer_state_on_cpu[k] = v.cpu()
        else:
            optimizer_state_on_cpu[k] = v

    checkpoint_payload = {
        'epoch': epoch_num,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer_state_on_cpu,
        'final_val_loss': metrics['final_val_loss'],
        'hyperparameters': {
            'alpha': criterion.alpha,
            'beta': criterion.beta,
            'gamma': criterion.gamma
        }
    }
    checkpoint_path = os.path.join(checkpoint_dir, f"PI_RAFT_epoch_{epoch_num}.pth")
    torch.save(checkpoint_payload, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

print("Initializing AdamW Optimizer...")
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

epochs = 3
step_counter = 0

for epoch in range(1, epochs + 1):
    model.train()
    print(f"\n--- Starting Training Epoch {epoch}/{epochs} ---")
    for batch_idx, (images_t, dem_t) in enumerate(sequence_loader):
        images_t, dem_t = images_t.to(device), dem_t.to(device)
        optimizer.zero_grad()
        flow_pred_t, height_pred_t = model(images_t, dem_t, iters=4)
        total_loss, data_loss, physics_loss = criterion(flow_pred_t, height_pred_t, images_t)
        total_loss.backward()
        optimizer.step()

        step_counter += 1
        if step_counter % 2 == 0:
            print(f"Step: {step_counter:03d} | TRAIN LOSS: {total_loss.item():.4f}")

    model.eval()
    val_total = 0.0
    with torch.no_grad():
        for images_v, dem_v in val_loader:
            images_v, dem_v = images_v.to(device), dem_v.to(device)
            flow_pred_v, height_pred_v = model(images_v, dem_v, iters=4)
            v_total, _, _ = criterion(flow_pred_v, height_pred_v, images_v)
            val_total += v_total.item()

    avg_val_total_loss = val_total / len(val_loader)
    print(f"VAL METRICS EPOCH {epoch} | Avg Loss: {avg_val_total_loss:.4f}")
    save_checkpoint(epoch, model, optimizer, {'final_val_loss': avg_val_total_loss})

In [ ]:
print("Executing Inference Iteration Scaling Analysis...")
model.eval()
images_val, dem_val = next(iter(val_loader))
images_val, dem_val = images_val.to(device), dem_val.to(device)

with torch.no_grad():
    flow_low_res, _ = model(images_val, dem_val, iters=4)
    flow_high_res, height_pred_val = model(images_val, dem_val, iters=12)

plot_inference_scaling_comparison(images_val[:, 0], flow_low_res, flow_high_res, standard_iters=4, production_iters=12)

In [ ]:
print("Initializing Meteorological NetCDF4 Export Engine...")
u_array = flow_high_res.squeeze(0)[0].cpu().numpy()
v_array = flow_high_res.squeeze(0)[1].cpu().numpy()
p_array = height_pred_val.squeeze(0)[0].cpu().numpy()

output_dir = "./outputs/meteorological_outputs"
os.makedirs(output_dir, exist_ok=True)
nc_filepath = os.path.join(output_dir, "INSAT_3DS_AMV_Derived_Wind.nc")

export_amv_netcdf(u_array, v_array, p_array, nc_filepath)

ds = xr.open_dataset(nc_filepath)
print(ds)

plot_netcdf_verification(nc_filepath)